## Chain rule

The chain rule tells you how to differentiate a **composition** of functions.

If `y = f(g(x))`, set `u = g(x)`. Then:

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

In words: **multiply the derivatives along the chain.** The rate at which `y` changes
with `x` is the rate `y` changes with `u`, times the rate `u` changes with `x`.

For a longer chain `y = f(g(h(x)))` you just keep multiplying:

$$\frac{dy}{dx} = \frac{dy}{du}\cdot\frac{du}{dv}\cdot\frac{dv}{dx}$$

**Why it matters for us:** backprop *is* the chain rule. A neural net is one big
composition of functions; to get the gradient of the loss w.r.t. any weight, you
multiply local derivatives backward from the loss to that weight.

In [5]:
import numpy as np

# Example: y = sin(x**2).  Let u = x**2, so y = sin(u).
#   dy/du = cos(u) = cos(x**2)
#   du/dx = 2x
#   dy/dx = cos(x**2) * 2x     <-- chain rule

def y(x):
    return np.sin(x**2)

def dy_dx_analytic(x):
    return np.cos(x**2) * 2*x

# Numerical check via central finite difference: (f(x+h) - f(x-h)) / 2h
def dy_dx_numeric(x, h=1e-5):
    return (y(x + h) - y(x - h)) / (2*h)

x = 1.3
print('analytic:', dy_dx_analytic(x))
print('numeric :', dy_dx_numeric(x))

analytic: -0.3091960801711924
numeric : -0.3091960803947025


## Partial derivatives

For a function of **several variables**, there's no single slope — it depends on which
direction you move. So we take the derivative **one variable at a time**, holding the
others fixed. The curly `∂` (vs. `d`) is the reminder that "everything else is frozen":

$$\frac{\partial f}{\partial x} = \text{rate } f \text{ changes as } x \text{ moves, with all other inputs held constant}$$

**Example:** `f(x, z) = x**2 * z + sin(z)`

- `∂f/∂x`: treat `z` as constant → `2xz`  (the `sin(z)` term is constant in `x`, so → 0)
- `∂f/∂z`: treat `x` as constant → `x**2 + cos(z)`

**The gradient** stacks all partials into a vector, pointing in the direction of
steepest increase (gradient descent steps along its negative):

$$\nabla f = \left[\frac{\partial f}{\partial x},\ \frac{\partial f}{\partial z}\right]$$

**Why it matters for us:** the loss depends on millions of weights. The gradient is the
vector of partials w.r.t. every weight — each computed by the chain rule while holding
the others fixed. That whole vector is what backprop produces.

In [6]:
# f(x, z) = x**2 * z + sin(z)
def f(x, z):
    return x**2 * z + np.sin(z)

# Analytic partials
def grad_analytic(x, z):
    df_dx = 2*x*z
    df_dz = x**2 + np.cos(z)
    return np.array([df_dx, df_dz])

# Numeric gradient: central difference, perturbing ONE variable at a time.
def grad_numeric(x, z, h=1e-5):
    df_dx = (f(x + h, z) - f(x - h, z)) / (2*h)   # z held fixed
    df_dz = (f(x, z + h) - f(x, z - h)) / (2*h)   # x held fixed
    return np.array([df_dx, df_dz])

x, z = 1.3, 0.7
print('analytic grad:', grad_analytic(x, z))
print('numeric  grad:', grad_numeric(x, z))

analytic grad: [1.82       2.45484219]
numeric  grad: [1.82       2.45484219]


## Gradients of scalar functions

A **scalar function** maps a vector to a single number: `f: ℝⁿ → ℝ`. Think of the loss:
many inputs (all the weights), one output (the loss value).

Its **gradient** is the vector holding the partial derivative w.r.t. *each* input:

$$\nabla f(\mathbf{x}) = \left[\frac{\partial f}{\partial x_1},\ \frac{\partial f}{\partial x_2},\ \dots,\ \frac{\partial f}{\partial x_n}\right]$$

Key facts:

- **Same shape as the input.** If `x` is a length-`n` vector, `∇f` is also length `n`.
  (If `x` is a matrix of weights, `∇f` is a matrix of the same shape.)
- **Direction of steepest ascent.** `∇f` points where `f` increases fastest; `−∇f` is the
  descent direction. Gradient descent is literally `x ← x − lr · ∇f`.
- **Zero at flat points.** At a minimum/maximum/saddle, `∇f = 0`.

**The numeric gradient checker (generalized).** The per-variable loop from before becomes:
perturb each component `x_i` by `±h`, hold the rest fixed, take a central difference. This
gives a slow but formula-free gradient — the tool we'll use to verify every backward pass
in this project.

In [7]:
# A scalar function of a VECTOR:  f(x) = sum_i (x_i**2)  +  x_0 * x_1
#   -> maps R^n to a single number.
def f(x):
    return np.sum(x**2) + x[0] * x[1]

# Analytic gradient (worked out by hand, component by component):
#   d/dx_i of sum(x^2) = 2 x_i
#   the extra x0*x1 term adds x1 to component 0, and x0 to component 1
def grad_analytic(x):
    g = 2 * x.copy()
    g[0] += x[1]
    g[1] += x[0]
    return g

# General numeric gradient checker: loops over EVERY component,
# perturbs just that one by +/- h, central difference. Works for any f: R^n -> R.
def numeric_gradient(f, x, h=1e-5):
    grad = np.zeros_like(x, dtype=float)
    for i in range(x.size):
        step = np.zeros_like(x, dtype=float)
        step[i] = h
        grad[i] = (f(x + step) - f(x - step)) / (2*h)
    return grad

x = np.array([1.0, 2.0, -3.0, 0.5])
print('analytic:', grad_analytic(x))
print('numeric :', numeric_gradient(f, x))
print('max abs diff:', np.max(np.abs(grad_analytic(x) - numeric_gradient(f, x))))

analytic: [ 4.  5. -6.  1.]
numeric : [ 4.  5. -6.  1.]
max abs diff: 1.2812417793384157e-10


## Softmax

**Softmax** turns a vector of arbitrary real numbers (**logits**) into a **probability
distribution** — all entries in `(0, 1)` and summing to `1`:

$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

How to read it:

- **Exponentiate** every logit → all values become positive.
- **Normalize** by the sum → they become probabilities that add to 1.
- It's **monotonic**: the largest logit gets the largest probability. It's a *soft* argmax —
  instead of picking one winner, it spreads weight, mostly onto the top entries.

Properties that matter later:

- **Shift-invariant:** adding a constant to every logit doesn't change the output (this is
  exactly what the log-sum-exp trick exploits below).
- **Not scale-invariant:** multiplying logits by a factor sharpens (large factor → near
  one-hot) or flattens (small factor → near uniform) the distribution. That factor is the
  **temperature** knob used in sampling.

**Why it matters for us:** it's the output layer of a classifier / language model — it
converts the network's raw scores into `p(class)` or `p(next token)`, and it pairs with
cross-entropy loss (Step 2).

In [9]:
# Softmax: logits -> probability distribution
def softmax(x):
    e = np.exp(x - np.max(x))   # stable form (see next section for why)
    return e / np.sum(e)

logits = np.array([2.0, 1.0, 0.1])
p = softmax(logits)
print('probs      :', p)
print('sum to 1   :', p.sum())
print('argmax kept:', np.argmax(logits) == np.argmax(p))   # largest logit -> largest prob

# Temperature: scaling logits sharpens or flattens the distribution
for T in [0.5, 1.0, 5.0]:
    print(f'T={T}: {np.round(softmax(logits / T), 3)}')   # small T -> peaky, large T -> flat

probs      : [0.65900114 0.24243297 0.09856589]
sum to 1   : 1.0
argmax kept: True
T=0.5: [0.864 0.117 0.019]
T=1.0: [0.659 0.242 0.099]
T=5.0: [0.4   0.327 0.273]


## Numerical stability: floating point, overflow, log-sum-exp

### Floating point
Computers store reals in finite bits (float64 = 64 bits). Two consequences:

- **Finite range.** The largest float64 is about `1.8e308`. Anything bigger becomes `inf`
  (**overflow**); anything tiny underflows to `0.0`.
- **Finite precision.** Only ~15–16 significant decimal digits. `0.1 + 0.2 != 0.3` exactly,
  and subtracting two nearly-equal numbers throws away digits (**catastrophic cancellation**).

### Where it bites us: softmax
Softmax needs exponentials: `softmax(x)_i = exp(x_i) / Σ_j exp(x_j)`. But `exp` grows
insanely fast — `exp(1000)` is `inf` in float64. Then `inf / inf = nan` and the whole
forward pass is poisoned. Logits in a real net easily reach these ranges.

### The log-sum-exp trick
Softmax is **shift-invariant**: subtracting the same constant `c` from every logit leaves
the result unchanged, because the constant cancels top and bottom:

$$\frac{e^{x_i - c}}{\sum_j e^{x_j - c}} = \frac{e^{-c}\,e^{x_i}}{e^{-c}\sum_j e^{x_j}} = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Choose `c = max(x)`. Now the largest exponent is `exp(0) = 1` — no overflow — and every
other term is between 0 and 1. Same math, safe numbers.

The same idea gives a stable **log-sum-exp**, `log Σ exp(x_j) = c + log Σ exp(x_j − c)`,
which is how cross-entropy loss is computed directly from logits (never forming the raw
`exp`).

In [10]:
# Floating-point limits
print('largest float64 :', np.finfo(np.float64).max)
print('exp(1000)       :', np.exp(1000.0))      # -> inf (overflow)
print('0.1 + 0.2 == 0.3:', 0.1 + 0.2 == 0.3)    # -> False (finite precision)

# Naive softmax: overflows on large logits
def softmax_naive(x):
    e = np.exp(x)
    return e / np.sum(e)

# Stable softmax: subtract the max first (log-sum-exp trick)
def softmax_stable(x):
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

x = np.array([1000.0, 1001.0, 1002.0])   # big logits
print('\nnaive :', softmax_naive(x))     # -> [nan nan nan]
print('stable:', softmax_stable(x))      # -> correct, finite

# Same answer as naive on SMALL inputs (shift-invariance, sanity check)
small = np.array([1.0, 2.0, 3.0])
print('\nagree on small inputs:', np.allclose(softmax_naive(small), softmax_stable(small)))

# Stable log-sum-exp: log(sum(exp(x))) without ever forming exp(x) directly
def logsumexp(x):
    c = np.max(x)
    return c + np.log(np.sum(np.exp(x - c)))

print('logsumexp(big logits):', logsumexp(x))   # finite, ~1002.4

largest float64 : 1.7976931348623157e+308
exp(1000)       : inf
0.1 + 0.2 == 0.3: False

naive : [nan nan nan]
stable: [0.09003057 0.24472847 0.66524096]

agree on small inputs: True
logsumexp(big logits): 1002.4076059644444


/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:3: RuntimeWarning: overflow encountered in exp
  This is separate from the ipykernel package so we can avoid doing imports until
/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:8: RuntimeWarning: overflow encountered in exp
  
/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/ipykernel_launcher.py:9: RuntimeWarning: invalid value encountered in true_divide
  if __name__ == "__main__":


## Backprop

Training = minimize a scalar loss `L` over millions of parameters via gradient descent,
which needs `∂L/∂θ` for every parameter `θ`. **Backprop** computes all of them exactly in
**one forward + one backward pass** (vs. `numeric_gradient`, which needs 2 passes *per
parameter*).

It's the **chain rule applied backward** through the network:

- **Forward:** run input → output, caching intermediate values.
- **Backward:** start at the loss (`∂L/∂L = 1`) and walk backward. Each layer receives the
  upstream gradient `∂L/∂(its output)` and returns `∂L/∂(params)` (to update) and
  `∂L/∂(its input)` (passed to the previous layer).

Efficient because the loss is a single scalar, so one backward sweep reaches every
parameter (reverse-mode autodiff). `numeric_gradient` stays as the checker that verifies
each hand-derived backward is correct.

## Activations: sigmoid, tanh, ReLU, GELU

Without a nonlinearity between linear layers, the whole stack collapses to one linear map.
Activations add the curvature that lets a net approximate non-linear functions.

| Name | Formula | Range | Derivative |
|---|---|---|---|
| **sigmoid** | `1 / (1 + e^-x)` | (0, 1) | `s(x)·(1 − s(x))` |
| **tanh** | `(e^x − e^-x)/(e^x + e^-x)` | (−1, 1) | `1 − tanh(x)²` |
| **ReLU** | `max(0, x)` | [0, ∞) | `1 if x>0 else 0` |
| **GELU** | `x · Φ(x)` (Φ = normal CDF) | ≈[−0.17, ∞) | smooth, ≈ReLU |

- **sigmoid / tanh** squash into a fixed range but **saturate** (flat tails → tiny
  gradients). tanh is zero-centered, usually preferred over sigmoid for hidden layers.
- **ReLU** is cheap and doesn't saturate for `x>0` → the default for deep nets.
- **GELU** is a smooth ReLU used in Transformers (what we'll use).

In [ ]:
import numpy as np

def sigmoid(x):   return 1 / (1 + np.exp(-x))
def d_sigmoid(x): s = sigmoid(x); return s * (1 - s)

def tanh(x):      return np.tanh(x)
def d_tanh(x):    return 1 - np.tanh(x) ** 2

def relu(x):      return np.maximum(0, x)
def d_relu(x):    return (x > 0).astype(float)

# GELU (tanh approximation) and its derivative
def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

# Check a derivative numerically (central difference) at a point
x0 = 0.7
num = (sigmoid(x0 + 1e-5) - sigmoid(x0 - 1e-5)) / (2e-5)
print('d_sigmoid analytic:', d_sigmoid(x0))
print('d_sigmoid numeric :', num)

## Weight initialization (why scale matters: Xavier/He)

Weights start random (to break symmetry) but their **scale** is critical. Each layer
multiplies its input by `W`; if `W` is too large the signal (and gradients) **explode**
across layers, too small and they **vanish**.

The fix: scale the initial variance by the layer width so signal magnitude is preserved
layer to layer:

- **Xavier/Glorot** — `Var(W) = 1 / n_in` (or `2/(n_in+n_out)`). For tanh/sigmoid.
- **He** — `Var(W) = 2 / n_in`. For ReLU (accounts for ReLU zeroing half the inputs).

Practically: `W = randn(n_in, n_out) * sqrt(scale / n_in)`.

## Vanishing / exploding gradients

Backprop multiplies many local derivatives along the path from loss to an early layer. If
those factors are consistently **< 1**, the product shrinks toward 0 (**vanishing** — early
layers barely learn); if **> 1**, it blows up (**exploding** — unstable, `nan`).

This is why deep nets were historically hard to train. Mitigations we'll use:
- **ReLU/GELU** (derivative 1 for active units, doesn't saturate)
- **good init** (Xavier/He keeps factors near 1)
- **residual connections** and **LayerNorm** (Step 5) — keep gradients flowing through depth.

## Universal approximation theorem

A feedforward net with a **single hidden layer** and a nonlinear activation can approximate
*any* continuous function on a bounded region, to arbitrary accuracy — given enough hidden
units.

Caveats worth knowing:
- It's an **existence** result: it says such weights exist, not that gradient descent will
  find them.
- "Enough units" can be impractically large. **Depth** (many layers) expresses many
  functions far more efficiently than one very wide layer — which is why we go deep.

## Dead ReLUs & activation saturation

Two failure modes where neurons stop learning because their **gradient is ~0**:

- **Dead ReLU** — if a ReLU's input is always negative, its output is always 0 and its
  derivative is always 0, so no gradient flows and the weights never update — the neuron is
  permanently "dead." Caused by bad init or too-large learning rates. Mitigated by careful
  init, lower lr, or variants (LeakyReLU, GELU).
- **Saturation** — sigmoid/tanh have flat tails; for large `|x|` the derivative → 0, so
  gradients vanish there too. Another reason ReLU-family activations are preferred in the
  hidden layers of deep nets.

## Cross-entropy loss

The loss for classification. The model outputs probabilities `p = softmax(logits)` over
classes; the true label is a class index (equivalently a **one-hot** vector). Cross-entropy
measures how far `p` is from the truth:

$$L = -\sum_c \text{onehot}_c \, \log p_c = -\log p_{\text{correct}}$$

Since one-hot is 0 everywhere except the true class, it collapses to **just the negative log
of the probability assigned to the correct class**.

### Intuition — it rewards confident-correct, punishes confident-wrong

The `-log` shape is the whole story:

| p(correct) | `-log p` = loss | meaning |
|---|---|---|
| 1.0 | 0.00 | perfect, no loss |
| 0.9 | 0.11 | confident & right → tiny loss |
| 0.5 | 0.69 | unsure → moderate loss |
| 0.1 | 2.30 | confident & **wrong** → big loss |
| 0.01 | 4.61 | very confident & wrong → huge loss |
| → 0 | → ∞ | assigning ~0 to the truth is catastrophic |

So it isn't enough to rank the right class highest — the model is pushed to put **high
probability** on it. Being confidently wrong is punished far more than being unsure.

### Worked example (3 classes, true class = 0)

- `p = [0.9, 0.05, 0.05]` → `L = -log(0.9) = 0.105`  (good)
- `p = [0.5, 0.3, 0.2]`  → `L = -log(0.5) = 0.693`  (unsure)
- `p = [0.1, 0.6, 0.3]`  → `L = -log(0.1) = 2.303`  (confident, wrong)

Only `p[0]` (the true class) enters the loss — the other entries matter only because
softmax makes them compete for the same probability mass.

### Why `log`

Log turns "probability of getting everything right" (a product) into a **sum** of per-example
terms — nicer to optimize — and its explosion near 0 (`-log(x) → ∞`) is what makes the model
*terrified* of assigning near-zero probability to the truth.

### Batched form

Average the per-example loss over the batch:  `L = mean over examples of -log p_correct`.

### The clean combined gradient

Softmax and cross-entropy are derived *together*, because the messy pieces cancel into one
strikingly simple result:

$$\frac{\partial L}{\partial \text{logits}} = p - \text{onehot} = \text{softmax} - \text{onehot}$$

Read it as **"predicted distribution minus true distribution"**. Example (true class 0,
`p = [0.9, 0.05, 0.05]`):

$$p - \text{onehot} = [0.9-1,\; 0.05,\; 0.05] = [-0.1,\; +0.05,\; +0.05]$$

Negative on the correct class (gradient descent will **raise** that logit), positive on the
wrong ones (**lower** them). This is why the output layer's backward is a one-liner — you
never differentiate softmax and log separately. Compute it straight from **logits** (with the
stable log-sum-exp) for numerical safety.

In [ ]:
import numpy as np

def softmax(x):
    e = np.exp(x - np.max(x)); return e / e.sum()

def cross_entropy(logits, target):      # target = correct class index
    return -np.log(softmax(logits)[target])

# --- how loss depends on confidence in the CORRECT class (true = 0) ---
print('loss vs. probability on the correct class:')
for p_correct in [1.0, 0.9, 0.5, 0.1, 0.01]:
    print(f'  p(correct)={p_correct:>4}  ->  loss = {-np.log(p_correct):.3f}')

# --- three predictions, same true class 0 ---
print('\nthree predictions (true class = 0):')
for name, probs in [('good', [0.9,0.05,0.05]),
                    ('unsure', [0.5,0.3,0.2]),
                    ('confident-wrong', [0.1,0.6,0.3])]:
    print(f'  {name:16s} p={probs}  loss={-np.log(probs[0]):.3f}')

# --- the combined gradient  softmax - onehot ---
logits = np.array([2.0, 0.5, -1.0, 1.0]); target = 2
onehot = np.zeros_like(logits); onehot[target] = 1.0
analytic = softmax(logits) - onehot
numeric = np.array([
    (cross_entropy(logits + e, target) - cross_entropy(logits - e, target)) / 2e-5
    for e in (np.eye(len(logits)) * 1e-5)
])
print('\ngradient (softmax - onehot):')
print('  analytic:', analytic.round(4))
print('  numeric :', numeric.round(4), '  <- matches')

## Gradient descent: SGD, learning rate, mini-batches

Weights move downhill against the gradient:

$$\theta \leftarrow \theta - \eta \, \nabla_\theta L$$

- **Learning rate `η`** — the step size. Too small → painfully slow; too large → overshoots,
  loss oscillates or diverges. The single most important hyperparameter.
- **Mini-batches** — estimate the gradient from a small chunk of data per step (instead of the
  whole dataset), giving many more updates per pass. Some noise, which is usually fine.
- **SGD** (stochastic gradient descent) — the name for gradient descent using mini-batch
  (or single-example) gradient estimates. The baseline optimizer everything else builds on.

## Adam

Plain SGD uses **one fixed learning rate for every parameter**. That struggles when different
parameters need different step sizes, or when gradients are noisy. **Adam** fixes this by
tracking two running averages of the gradient, **per parameter**.

### The two moments

- **`m` — 1st moment (momentum):** an exponential moving average of recent gradients.
  Smooths the direction and carries velocity through flat or noisy regions (like a ball
  rolling downhill instead of reacting to every bump).
- **`v` — 2nd moment (variance):** an exponential moving average of recent *squared*
  gradients. Used to scale the step **per parameter**: a parameter with consistently large
  gradients gets a *smaller* effective step, a rarely-updated one gets a *larger* step.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2$$

$$\hat m_t = \frac{m_t}{1-\beta_1^{\,t}}, \quad \hat v_t = \frac{v_t}{1-\beta_2^{\,t}}, \qquad
\theta_t = \theta_{t-1} - \eta \, \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}$$

### Why bias correction

`m` and `v` start at **0**, so the first few averages are biased toward 0 (too small). Dividing
by `(1 - \beta^t)` undoes this. Concretely, at step `t=1` with `g=1`, `\beta_1=0.9`:

- raw `m = 0.9·0 + 0.1·1 = 0.1`  → without correction the first step is ~10× too small
- corrected `\hat m = 0.1 / (1 - 0.9^1) = 0.1 / 0.1 = 1.0`  → back to the true gradient

As `t` grows, `\beta^t → 0` and the correction fades away (it only matters early).

### The effective step

`\hat m / (\sqrt{\hat v} + \epsilon)` is roughly **"direction, normalized by typical
magnitude"** — so the step size is close to `\eta` regardless of how big or small the raw
gradients are. That self-scaling is why Adam trains fast with little tuning.

### Defaults

`\beta_1 = 0.9`, `\beta_2 = 0.999`, `\epsilon = 1e-8`, `\eta ≈ 1e-3`. Robust out of the box —
which is why Adam (and AdamW) is the standard optimizer for Transformers.

In [ ]:
import numpy as np

# Minimize f(x) = x^2  (min at x=0, gradient = 2x).  Compare SGD vs Adam.
def sgd(lr=0.1, steps=40):
    x = 5.0
    for _ in range(steps):
        g = 2*x
        x -= lr*g
    return x

def adam(lr=0.1, steps=40, b1=0.9, b2=0.999, eps=1e-8):
    x = 5.0; m = 0.0; v = 0.0
    for t in range(1, steps+1):
        g = 2*x
        m = b1*m + (1-b1)*g          # 1st moment
        v = b2*v + (1-b2)*g*g        # 2nd moment
        mhat = m / (1 - b1**t)       # bias correction
        vhat = v / (1 - b2**t)
        x -= lr * mhat / (np.sqrt(vhat) + eps)
        if t in (1, 2, 5, 40):
            print(f'  step {t:2d}: x={x:+.4f}  m={m:+.3f}  v={v:.3f}  mhat={mhat:+.3f}')
    return x

print('Adam trajectory minimizing x^2 from x=5:')
xa = adam()
print(f'\nfinal x  -> SGD: {sgd():.4f}   Adam: {xa:.4f}   (target 0.0)')

## Optimizers landscape (good to know)

- **SGD + momentum** — add a velocity term to SGD; cheap, strong generalization (common in
  vision).
- **RMSProp** — per-parameter adaptive step from squared-gradient average (Adam's `v` half).
- **Adam** — momentum + RMSProp combined; the general-purpose default.
- **AdamW** — Adam with **decoupled weight decay** (L2 applied directly to weights, not folded
  into the gradient). The standard for training Transformers today.

## Learning-rate schedules (good to know)

The learning rate usually shouldn't stay constant:

- **Warmup** — start `η` tiny and ramp up over the first steps; avoids early instability while
  Adam's moment estimates are still noisy.
- **Decay** — shrink `η` over training (cosine or linear) so steps get finer as you near a
  minimum. **Warmup-then-cosine-decay** is the typical Transformer recipe.

## Regularization (good to know)

Techniques that fight **overfitting** (memorizing train data instead of generalizing):

- **L2 / weight decay** — penalize large weights → simpler, smoother models.
- **Dropout** — randomly zero some activations during training → prevents co-dependence.
- **Early stopping** — stop when *validation* loss stops improving.
- **Label smoothing** — soften one-hot targets (e.g. 0.9/0.1) → less overconfident, better
  calibrated.

## Metrics: accuracy vs. loss vs. perplexity (good to know)

- **Loss** — what you optimize (cross-entropy); smooth and differentiable, but not intuitive.
- **Accuracy** — fraction of correct predictions; intuitive, but not differentiable (can't
  train on it directly) and coarse.
- **Perplexity** — the language-model metric, `exp(cross-entropy)`. Read it as "on average the
  model is this unsure among how many choices" — perplexity 1 = perfect, `vocab_size` = random
  guessing. We'll use it in Step 8.

# Step 3 — Tokenizer, Embeddings & Language Modeling

Everything below is the language machinery our captioning model reuses: how text becomes
numbers, how numbers become learnable vectors, and what "predict the next token" means.

## Language modeling: next-token prediction

A **language model** assigns probability to a sequence of tokens, and — the useful part —
predicts the **next token** given everything before it:

$$p(w_t \mid w_1, w_2, \dots, w_{t-1})$$

By the **chain rule of probability**, the probability of a whole sequence factorizes into a
product of next-token probabilities:

$$p(w_1, \dots, w_T) = \prod_{t=1}^{T} p(w_t \mid w_{<t})$$

This is **autoregressive**: generate left to right, each step conditioning on the tokens
already produced. Training maximizes the probability of the real next token at every
position — equivalently, **minimizes cross-entropy** (Step 2) between the predicted
distribution and the actual next token (a one-hot).

- **Objective per position:** "predict next token" is a classification over the vocabulary.
  Loss = cross-entropy(logits_t, w_t), averaged over all positions.
- **Perplexity** = exp(average cross-entropy): roughly "how many choices the model is
  effectively unsure among" (see the Step 2 metrics cell).

**Why it matters for us:** our captioning model *is* this — next-token prediction over caption
tokens, conditioned on image tokens. Steps 4–5 add attention so the conditioning becomes
powerful.

In [ ]:
import numpy as np

# Toy 3-token vocab {0:'a', 1:'b', 2:'.'}. A model gives next-token distributions
# (rows sum to 1) that depend on the current token:  p(next | current)
P = {
    0: np.array([0.1, 0.7, 0.2]),   # after 'a'
    1: np.array([0.6, 0.1, 0.3]),   # after 'b'
    2: np.array([0.5, 0.4, 0.1]),   # after '.'
}
itos = {0: 'a', 1: 'b', 2: '.'}

seq = [0, 1, 2]                      # the sequence "a b ."
logp = 0.0
for current, nxt in zip(seq[:-1], seq[1:]):
    p = P[current][nxt]
    logp += np.log(p)
    print(f"p({itos[nxt]} | {itos[current]}) = {p}")

n = len(seq) - 1
print("sequence probability :", np.exp(logp))       # product of the conditionals
print("avg cross-entropy    :", -logp / n)          # mean negative log-likelihood
print("perplexity           :", np.exp(-logp / n))  # exp(cross-entropy)

## Tokenization

Models work with **integers**, not raw text. A **tokenizer** converts between text and a
sequence of integer **token ids**, using a fixed **vocabulary** (the set of known tokens).

- **encode:** `"hi"` -> `[7, 12]`
- **decode:** `[7, 12]` -> `"hi"`

**Granularity** — what counts as one token:

| Level | Token = | Vocab size | Sequence length | Notes |
|---|---|---|---|---|
| **character** | one character | tiny (~100) | long | trivial to build; no `<unk>` if all chars seen |
| **word** | one word | large (10k-100k+) | short | needs `<unk>` for unseen words |
| **sub-word (BPE)** | frequent chunks | medium (~30k-50k) | medium | best of both; what real LLMs use |

**Special tokens** — reserved ids with structural meaning:

- `<bos>` — beginning of sequence (the "start generating" signal)
- `<eos>` — end of sequence (tells generation to stop)
- `<pad>` — filler so batched sequences share a length (masked out of the loss)
- `<unk>` — unknown token, for inputs outside the vocabulary

**Why it matters for us:** we build a **character-level** tokenizer (simplest, no `<unk>`), add
`<bos>`/`<eos>` so the model knows where a caption starts and stops, and later `<pad>` to batch
captions of different lengths.

In [ ]:
import numpy as np

class CharTokenizer:
    # Character-level tokenizer: text <-> list of integer ids.

    def __init__(self, text: str) -> None:
        specialTokens: list = ["<pad>", "<bos>", "<eos>"]
        uniqueCharacters: list = sorted(set(text))
        self.idToToken: list = specialTokens + uniqueCharacters
        self.tokenToId: dict = {token: i for i, token in enumerate(self.idToToken)}

    @property
    def vocabSize(self) -> int:
        return len(self.idToToken)

    def encode(self, text: str, addSpecials: bool = True) -> list:
        tokenIds: list = [self.tokenToId[character] for character in text]
        if addSpecials:
            tokenIds = [self.tokenToId["<bos>"]] + tokenIds + [self.tokenToId["<eos>"]]
        return tokenIds

    def decode(self, tokenIds: list, skipSpecials: bool = True) -> str:
        specials: set = {"<pad>", "<bos>", "<eos>"}
        return "".join(
            self.idToToken[i] for i in tokenIds
            if not (skipSpecials and self.idToToken[i] in specials)
        )

tokenizer = CharTokenizer("a dog runs")
print("vocab size :", tokenizer.vocabSize)
encoded = tokenizer.encode("a dog")
print("encode     :", encoded)
print("decode     :", repr(tokenizer.decode(encoded)))
print("round-trip :", tokenizer.decode(tokenizer.encode("a dog")) == "a dog")

## Sub-word tokenization: BPE, WordPiece, SentencePiece (good to know)

Character-level makes long sequences; word-level makes huge vocabularies and chokes on unseen
words. **Sub-word** tokenizers split text into frequent chunks — common words stay whole, rare
words break into pieces:

- **BPE (Byte-Pair Encoding)** — start from characters, repeatedly merge the most frequent
  adjacent pair into a new token until the vocab hits a target size. GPT uses byte-level BPE
  (`tiktoken`).
- **WordPiece** — like BPE but merges by likelihood gain rather than raw frequency (BERT).
- **SentencePiece** — trains directly on raw text (no pre-tokenization); language-agnostic.

Benefits: fixed vocab, no `<unk>` (anything decomposes to sub-words/bytes), shorter sequences
than char-level. The assignment permits an existing tokenizer (e.g. `tiktoken`) — but
char-level is enough for our small corpus and keeps everything from scratch.

## Embeddings

A token id is just an integer label — no meaning, no notion of "close to." An **embedding**
maps each id to a **learned dense vector**, so the model can represent similarity and
structure:

$$\text{id } i \;\longrightarrow\; E[i] \in \mathbb{R}^{d}$$

The **embedding table** `E` has shape `(vocab_size, d)` — one row per token. "Embedding a
token" is just **selecting its row**: `E[i]`.

### It equals one-hot @ E, but cheaper
Selecting row `i` equals multiplying a one-hot vector (1 at position `i`) by `E`:

$$\text{onehot}(i)\, E = E[i]$$

So a lookup is a matmul with a one-hot input — but we skip building the one-hot and just
**index**. Same result, far cheaper.

- The table `E` is a **trainable parameter**; its rows are learned by backprop like any weight.
- After training, similar tokens end up with nearby vectors (see Word2Vec below).

**Why it matters for us:** we have a **token embedding** for caption tokens, and the **patch
embedding** projects image patches into the *same* `d`-dim space so attention can mix them.
Positions get embeddings too (below).

In [1]:
import numpy as np

vocabSize, d = 6, 4
rng = np.random.default_rng(0)
embeddingTable = rng.standard_normal((vocabSize, d))   # one row per token

tokenIds = np.array([2, 0, 4, 2])          # a sequence of ids
embedded = embeddingTable[tokenIds]        # lookup -> shape (len, d)
print("embedded shape:", embedded.shape)

# Equivalence: one-hot @ E selects the same rows
oneHot = np.zeros((len(tokenIds), vocabSize))
oneHot[np.arange(len(tokenIds)), tokenIds] = 1
print("lookup == onehot @ E:", np.allclose(embedded, oneHot @ embeddingTable))

embedded shape: (4, 4)
lookup == onehot @ E: True


## Embedding backward: scatter-add

Forward is a lookup: `embedded[t] = E[ids[t]]`. So the gradient w.r.t. the table `E` only
touches the **rows that were used**, and each used row receives the upstream gradient of that
position:

$$\frac{\partial L}{\partial E[i]} = \sum_{t \,:\, \text{ids}[t]=i} \text{dEmbedded}[t]$$

The **sum** is the key: if a token id appears **multiple times**, its row gets a contribution
from *each* occurrence — they **add up**. This is a **scatter-add**: scatter each position's
gradient into the row of its id, accumulating on collisions.

- Naive `dE[ids] = dEmbedded` is **wrong** — it overwrites on duplicate ids instead of summing.
- Correct: `np.add.at(dE, ids, dEmbedded)` (or an explicit loop with `+=`).

**Why it matters for us:** captions repeat characters/words constantly (spaces, "a", "the"),
so this accumulation is the norm, not an edge case.

In [ ]:
import numpy as np

vocabSize, d = 5, 3
tokenIds = np.array([1, 3, 1, 1])                                  # id 1 appears 3x
dEmbedded = np.arange(1, 4 * d + 1, dtype=float).reshape(4, d)     # upstream grad per position

# WRONG: assignment overwrites duplicates (id 1 keeps only its last gradient)
dE_wrong = np.zeros((vocabSize, d))
dE_wrong[tokenIds] = dEmbedded

# RIGHT: scatter-add accumulates duplicates
dE_right = np.zeros((vocabSize, d))
np.add.at(dE_right, tokenIds, dEmbedded)

print("row for id=1, wrong :", dE_wrong[1])
print("row for id=1, right :", dE_right[1], "(sum of its 3 occurrences)")
print("manual sum          :", dEmbedded[[0, 2, 3]].sum(axis=0))

## Word2Vec & embedding geometry (good to know)

Trained embeddings arrange meaning **geometrically**:

- **Similar words are close** (by cosine similarity) — "cat" near "dog".
- **Directions encode relations** — the famous `king - man + woman ~= queen`: analogies become
  vector arithmetic.

**Word2Vec** (2013) learned such vectors from context (predict a word from its neighbors, or
vice versa). Our embeddings are learned end-to-end for the task rather than separately, but the
same geometry emerges — which is why the grounding analysis (Step 8) can ask whether a caption
word's representation lines up with the right image region.

## N-gram models and the bigram LM

Before neural nets, language models were **n-grams**: estimate `p(w_t | previous n-1 tokens)`
by **counting** occurrences in a corpus.

- **Markov assumption:** only the last `n-1` tokens matter (a fixed, short context).
- **bigram (n=2):** `p(w_t | w_{t-1})` — condition on just the previous token. The simplest
  possible LM.

**Counting bigrams:** build `counts[i, j]` = "how often token `j` follows token `i`", then
normalize each row to probabilities. No learning — pure statistics.

**Neural bigram (our version):** replace the count table with a tiny network — embed the
current token, project to vocab logits, softmax — trained with cross-entropy. It *learns* the
same conditional distribution, and unlike counts it **scales** to real context via attention
(Steps 4-5). The bigram is our bridge from "counting" to "a trained LM."

**Limits of n-grams:** context capped at `n-1`; the table grows as `vocab^n` (sparse,
memory-hungry); unseen n-grams need smoothing. Embeddings + attention lift all three limits —
the reason the field moved to neural models.

In [ ]:
import numpy as np

text = "abracadabra"
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
V = len(chars)

# Count bigrams: counts[i, j] = # times j follows i
counts = np.zeros((V, V))
for a, b in zip(text[:-1], text[1:]):
    counts[stoi[a], stoi[b]] += 1

# Row-normalize to p(next | current)
probs = counts / counts.sum(axis=1, keepdims=True)
print("chars        :", chars)
print("p(next | 'a'):", np.round(probs[stoi['a']], 2))

# Generate by sampling from the bigram distribution
rng = np.random.default_rng(0)
out = ['a']
for _ in range(10):
    current = stoi[out[-1]]
    nxt = rng.choice(V, p=probs[current])
    out.append(itos[nxt])
print("sample       :", "".join(out))

## Teacher forcing vs. autoregressive generation

The same model is used two ways; the input differs.

**Training — teacher forcing.** Feed the **real** sequence; the target at each position is the
**next real token**. Inputs and targets are the same sequence shifted by one:

```
tokens : <bos>  a    dog   runs
inputs : <bos>  a    dog   runs
targets:  a     dog  runs  <eos>
```

Every position trains in parallel (with a causal mask, Step 4) on the ground-truth prefix —
fast and stable, because the model always sees correct history.

**Generation — autoregressive.** No target exists; feed the model's **own** previous outputs:
predict a token, append it, feed it back, repeat until `<eos>`. Slower (one token at a time)
and errors can compound.

**Exposure bias** — the mismatch: at train time the model only sees *correct* prefixes, but at
generation it must handle its *own* (possibly wrong) prefixes. A known limitation; at our scale
we accept it.

**Why it matters for us:** training feeds ground-truth captions (teacher forcing); the demo
generates captions autoregressively from an image.

In [ ]:
import numpy as np

# ids for "<bos> a dog <eos>"  (1=<bos>, 2=<eos>, others=content)
tokens = np.array([1, 5, 6, 7, 2])
inputs  = tokens[:-1]      # everything but the last
targets = tokens[1:]       # everything but the first (shifted by one)
for i, t in zip(inputs, targets):
    print(f"input id {i}  ->  predict next id {t}")

## Positional embeddings

Attention (Step 4) treats its inputs as a **set** — no built-in sense of order. Without
position information, "dog bites man" and "man bites dog" look identical. **Positional
embeddings** inject order by adding a position-dependent vector to each token embedding:

$$x_t = E_{\text{token}}[\text{id}_t] + E_{\text{pos}}[t]$$

Common schemes:

- **Learned** — a trainable table `(max_len, d)`, one row per position. Simple; what we'll use.
- **Sinusoidal** — fixed sines/cosines at different frequencies (original Transformer). No
  parameters; extrapolates to unseen lengths.
- **RoPE (rotary)** — rotates Q/K by position; the modern variant (good to know).

Because token and positional vectors are **added**, they share the same `d`-dim space, and the
model learns to separate "what" (token) from "where" (position).

**Why it matters for us:** our sequence is `[image tokens | caption tokens]`; positional
embeddings let the model tell the first caption token from the third, and where the image block
ends.

In [ ]:
import numpy as np

def sinusoidal_positions(maxLen: int, d: int) -> np.ndarray:
    position = np.arange(maxLen)[:, None]        # (maxLen, 1)
    dimIndex = np.arange(d)[None, :]             # (1, d)
    angle = position / (10000 ** (2 * (dimIndex // 2) / d))
    pe = np.zeros((maxLen, d))
    pe[:, 0::2] = np.sin(angle[:, 0::2])         # even dims: sine
    pe[:, 1::2] = np.cos(angle[:, 1::2])         # odd dims: cosine
    return pe

pe = sinusoidal_positions(maxLen=6, d=8)
print("positional embedding shape:", pe.shape)
print("position 0:", np.round(pe[0], 2))
print("position 1:", np.round(pe[1], 2))
# usage: x = token_embeddings + pe[:sequence_length]

## Sampling strategies at generation (good to know)

Given the next-token distribution, how do we pick? The choice trades **coherence** vs.
**diversity**:

- **Greedy** — argmax every step. Deterministic; safe but repetitive/generic.
- **Temperature `T`** — divide logits by `T` before softmax. `T<1` sharpens (more confident),
  `T>1` flattens (more random), `T->0` = greedy.
- **Top-k** — keep only the `k` highest-probability tokens, renormalize, sample. Cuts the long
  tail of nonsense.
- **Top-p (nucleus)** — keep the smallest set of tokens whose cumulative probability >= `p`
  (adaptive size), renormalize, sample.
- **Beam search** — keep the `b` best partial sequences by total probability; better for tasks
  with a "right" answer (translation), less used for open generation.

**Why it matters for us:** captions come from sampling — greedy for a stable demo,
temperature/top-k to show diversity in the report.

In [ ]:
import numpy as np

def softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max()); return e / e.sum()

logits = np.array([2.0, 1.0, 0.5, -1.0, -2.0])

print("greedy pick:", int(np.argmax(logits)))
for T in [0.5, 1.0, 2.0]:
    print(f"T={T}: {np.round(softmax(logits / T), 3)}")   # temperature sharpen/flatten

def top_k_probs(logits: np.ndarray, k: int) -> np.ndarray:
    keep = np.argsort(logits)[::-1][:k]            # indices of the k largest
    masked = np.full_like(logits, -np.inf)
    masked[keep] = logits[keep]
    return softmax(masked)

print("top-2 probs:", np.round(top_k_probs(logits, 2), 3))

# Step 4 — Sequence Models: RNN → Attention

Step 3 gave us tokens, embeddings and the next-token objective, but the only model we had was
the **bigram** — context of exactly one token. This step is about models that actually *use*
history. We start with the **RNN** (the pre-2017 answer), understand precisely where it breaks,
and that failure is what motivates **attention**.

## Recurrent Neural Networks (RNN)

### 1. Which problem are we solving?

Coming from Step 3, our language model was a **bigram**: `p(w_t | w_{t-1})`. Two hard limits:

- **Fixed, tiny context.** An n-gram sees only the last `n-1` tokens. To predict the last word
  of *"the dog that chased the cat across the yard was ___"*, you need the subject from 9 tokens
  back. Extending `n` doesn't scale: the count table grows as `vocab^n`.
- **No parameter sharing across time.** A plain MLP over a fixed window learns a *separate*
  weight for "the token 3 positions ago". It can't reuse "what a verb looks like" at every
  position, and it can't accept a sequence of a length it never saw.

We want a model that (a) handles **variable-length** sequences, (b) carries information from
**arbitrarily far back**, and (c) uses the **same parameters at every time step**. That model is
the RNN: keep a **hidden state** — a running summary of everything seen so far — and update it
with one shared function, once per token.

$$\text{bigram: } p(w_t \mid w_{t-1}) \qquad\longrightarrow\qquad \text{RNN: } p(w_t \mid h_{t-1}), \quad h_{t-1} = f(w_{<t})$$

The whole prefix is compressed into one vector `h`. That compression is the RNN's power *and*,
as we'll see in §5, exactly its weakness.

### 2. Architecture and the math behind it

An RNN is a **loop**: one small network applied repeatedly, its output fed back as input.

```
        x₁          x₂          x₃            <- input token embeddings
        │           │           │
h₀ ──▶ [A] ──h₁──▶ [A] ──h₂──▶ [A] ──h₃──▶    <- SAME weights A at every step
        │           │           │
        y₁          y₂          y₃            <- per-step output (e.g. next-token logits)
```

Unrolled, it's a deep feedforward net whose depth equals the **sequence length** — and whose
layers all **share one set of weights**.

**The recurrence (hidden state update):**

$$h_t = \tanh\!\big(W_{xh}\,x_t \;+\; W_{hh}\,h_{t-1} \;+\; b_h\big)$$

- `x_t` ∈ ℝ^`d_in` — input at step `t` (for us: the token embedding from Step 3).
- `h_{t-1}` ∈ ℝ^`d_h` — hidden state from the previous step: **the memory**, a compressed
  summary of tokens `1..t-1`. Initialized `h_0 = 0`.
- `W_xh` ∈ ℝ^(`d_h × d_in`) — **input-to-hidden**: how much the *new* token influences memory.
- `W_hh` ∈ ℝ^(`d_h × d_h`) — **hidden-to-hidden**, the recurrent matrix: how the old memory is
  transformed/retained. This single matrix is applied `T` times — the source of both long-range
  memory and the vanishing/exploding problem (§5).
- `b_h` ∈ ℝ^`d_h` — bias.
- `tanh` — the nonlinearity; **bounds `h` to (−1, 1)**, which keeps the repeatedly-multiplied
  state from blowing up. Its derivative `1 − h²` is ≤ 1 everywhere — remember this for §5.
- **`t` does not appear in the weights**: the same `W_xh, W_hh, b_h` are reused at every step
  (**weight tying across time**). That's what makes variable length possible.

**The output head (per step):**

$$y_t = W_{hy}\,h_t + b_y \qquad\qquad p_t = \text{softmax}(y_t)$$

- `W_hy` ∈ ℝ^(`vocab × d_h`) — projects the hidden state to **logits** over the vocabulary.
- `softmax` → next-token distribution; loss is **cross-entropy** vs. the true next token
  (Step 2), averaged over all `T` positions:

$$L = \frac{1}{T}\sum_{t=1}^{T} -\log p_t[w_{t+1}]$$

**Training: Backpropagation Through Time (BPTT).** Unroll the loop, then run ordinary backprop
on the resulting deep net. Two consequences of weight sharing:

$$\frac{\partial L}{\partial W_{hh}} \;=\; \sum_{t=1}^{T} \frac{\partial L}{\partial h_t}\,\frac{\partial h_t}{\partial W_{hh}}$$

- The gradient of a shared weight is the **sum of its contributions at every time step** (same
  accumulate-on-reuse rule as the embedding scatter-add in Step 3).
- `∂L/∂h_t` has **two** sources: the loss at step `t` (through `W_hy`) **plus** the gradient
  flowing back from step `t+1` (through `W_hh`) — memory is used later, so blame arrives from
  later.

And the term that decides whether long-range learning works at all — the gradient travelling
from step `t` back to an earlier step `k`:

$$\frac{\partial h_t}{\partial h_k} \;=\; \prod_{i=k+1}^{t} \frac{\partial h_i}{\partial h_{i-1}} \;=\; \prod_{i=k+1}^{t} \operatorname{diag}\!\big(1 - h_i^2\big)\, W_{hh}$$

- It is a **product of `t − k` Jacobians** — one factor per step of distance.
- `diag(1 − h_i²)` — the `tanh` derivative, each entry in **(0, 1]** (and ≈0 when `h` saturates
  near ±1).
- `W_hh` appears **`t − k` times**, so its largest singular value `σ` is raised to that power:
  `σ < 1` → the product **decays exponentially** (*vanishing gradient* — the model literally
  cannot learn the dependency); `σ > 1` → it **explodes** to `inf`/`nan`.
- Exponential in **distance**: this is the precise, mathematical reason a plain RNN forgets.
  (Compare: attention connects position `t` to position `k` in **one** step — a path length of
  1 instead of `t − k`. That is the punchline of the next section.)

**Practical fixes** used with RNNs: **gradient clipping** (rescale the gradient if its norm
exceeds a threshold — cures exploding), **truncated BPTT** (backprop only `k` steps back to bound
cost), and gated cells — **LSTM/GRU** — which add an *additive* memory path (`c_t = f_t ⊙ c_{t-1}
+ i_t ⊙ g_t`) so gradients can flow across many steps without being multiplied by `W_hh` each
time. Gates mitigate vanishing; they don't remove the sequential bottleneck.

**The pipeline of calculations, in two scenarios.** Same recurrence both times — what differs
is what flows through it.

**(A) Learning (training).** The whole sequence is known upfront (teacher forcing):

- Take the caption, split it into `inputs` and `targets` shifted by one.
- Embed the input ids into `x_1 … x_T`.
- Run the loop forward, `t = 1 … T`: compute `h_t` from `x_t` and `h_{t-1}`, then the logits
  `y_t`. Keep every `x_t` and `h_t` — the backward pass needs them.
- Compute the loss: cross-entropy of `y_t` against `target_t`, averaged over all `T` positions.
- Run the loop backward, `t = T … 1`. Each step's hidden state gets blame from **two** places:
  its own output, and the step after it (memory is used later, so blame comes from later).
- Accumulate the gradients of the shared weights — every step adds into the *same* `W_xh`,
  `W_hh`, `W_hy` (this summing *is* weight sharing).
- Clip the gradient if its norm is too large, then take one Adam step. Repeat on the next batch.

**(B) Actual working (generation).** Nothing exists yet; the model writes it:

- Weights are frozen. No targets, no loss, no backward pass, nothing cached.
- Start with `h = 0` (for captioning: `h` = the image feature vector — that's *Show and Tell*)
  and feed `<bos>`.
- Compute the new `h`, overwriting the old one — it's no longer needed.
- Project `h` to logits, softmax to probabilities, pick a token (greedy / temperature / top-k).
- Emit that token and feed it **back in** as the next input.
- Repeat until `<eos>` or a length cap.

**The differences that matter:**

- **Input:** training feeds ground-truth tokens; generation feeds the model's own last output.
  That mismatch is **exposure bias** (Step 3).
- **Passes:** training is forward + backward; generation is forward only.
- **Memory:** training stores all `T` hidden states (why *truncated* BPTT exists); generation
  stores exactly one vector, no matter how long the output — the one place a plain RNN still
  beats attention, whose KV-cache keeps growing.
- **Parallelism:** neither can be parallelized across time — `h_t` needs `h_{t-1}`. Training a
  length-`T` sequence means `T` dependent steps. This is the bottleneck attention removes.

### 3. Applications (when to use)

Anything where the input is a **sequence** and order matters:

- **Language modeling / text generation** (char-RNN — the direct ancestor of what we're building)
- **Sequence classification**: sentiment, intent — read the sequence, classify from the final `h`
- **Sequence labeling**: POS tagging, NER — one output per step
- **Seq2seq**: translation, summarization (encoder RNN → decoder RNN; the 2014-2016 standard,
  and where attention was *invented* as a patch)
- **Time series & audio**: forecasting, speech recognition — where signals are naturally streamed
- **Image captioning** — *our task*: the classic **Show and Tell** (2015) fed a CNN image vector
  as `h_0` into an LSTM decoder. We do the same job with a Transformer instead.

**Still genuinely useful today** when: the sequence is **very long or unbounded** (streaming
audio/sensors — `O(1)` memory per step beats attention's `O(n²)`), latency per token must be
tiny on-device, or the dataset is small. Otherwise, Transformers win.

### 4. Advantages

- **Variable-length input** — the loop just runs longer; no fixed window.
- **Constant parameter count** — independent of sequence length (weights are shared across time).
- **In principle unbounded context** — `h_t` can carry information from any earlier step.
- **`O(1)` memory / `O(n)` compute** in sequence length — cheap and **streaming-friendly**:
  process a token, update `h`, discard the token.
- **Inductive bias for order** — recency and sequence structure are built in; no positional
  encoding needed (contrast Step 3's positional embeddings, which attention *requires*).

### 5. Disadvantages

- **Vanishing / exploding gradients** — the `∏ W_hh` product of §2. Learning dependencies beyond
  ~10-20 steps is unreliable; LSTM/GRU push this out but don't solve it.
- **Inherently sequential — cannot be parallelized across time.** `h_t` needs `h_{t-1}`, so
  training on a length-`T` sequence takes `T` dependent steps and can't use a GPU's parallelism.
  **This, not accuracy, is the main reason Transformers replaced RNNs**: attention computes all
  positions at once.
- **Information bottleneck** — the entire prefix must fit in one fixed-size vector `h`. In
  seq2seq, the whole source sentence squeezed into one vector was the failure that attention was
  invented to fix.
- **Recency bias** — recent tokens dominate `h`; distant ones fade.
- **No direct access to the past** — to use token `k` at step `t`, the information must survive
  `t − k` lossy overwrites. Attention *reads token `k` directly*.
- **Hard to interpret** — no equivalent of an attention map showing what the model looked at.

> **The bridge to Step 4's real content:** every disadvantage above is about the *sequential,
> compressed* path between positions. Attention replaces it with a **direct, parallel, weighted
> lookup** over all positions at once — same goal (use the context), opposite mechanism.

### 6. Useful resources

- Karpathy — **The Unreasonable Effectiveness of Recurrent Neural Networks**
  <http://karpathy.github.io/2015/05/21/rnn-effectiveness/> — the char-RNN post: the same
  architecture as above, ~100 lines of NumPy, trained on Shakespeare / C code / LaTeX. Read the
  generated samples, then his `min-char-rnn.py` gist for a full forward + BPTT in one file.
- Olah — **Understanding LSTM Networks** <https://colah.github.io/posts/2015-08-Understanding-LSTMs/>
  — the canonical picture of why gates fix the vanishing product in §2.
- Bahdanau et al. (2014) — *Neural Machine Translation by Jointly Learning to Align and Translate*
  — attention invented as a patch for the RNN bottleneck (§5). The direct prequel to Step 4.

In [1]:
import numpy as np

# Minimal RNN: forward over a sequence, one shared set of weights reused at every step.
class SimpleRNN:
    def __init__(self, inputSize: int, hiddenSize: int, outputSize: int, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        self.hiddenSize: int = hiddenSize
        # Xavier-ish scaling (Step 1) so the repeated multiplications stay well behaved
        self.inputToHidden: np.ndarray  = rng.standard_normal((hiddenSize, inputSize))  / np.sqrt(inputSize)   # W_xh
        self.hiddenToHidden: np.ndarray = rng.standard_normal((hiddenSize, hiddenSize)) / np.sqrt(hiddenSize)  # W_hh
        self.hiddenBias: np.ndarray     = np.zeros(hiddenSize)                                                  # b_h
        self.hiddenToOutput: np.ndarray = rng.standard_normal((outputSize, hiddenSize)) / np.sqrt(hiddenSize)   # W_hy
        self.outputBias: np.ndarray     = np.zeros(outputSize)                                                  # b_y

    def forward(self, inputs: np.ndarray) -> tuple:
        # inputs: (T, inputSize) -- one embedding per time step
        sequenceLength: int = inputs.shape[0]
        hiddenState: np.ndarray = np.zeros(self.hiddenSize)      # h_0 = 0
        hiddenStates: list = []
        logits: list = []
        for t in range(sequenceLength):
            # h_t = tanh(W_xh x_t + W_hh h_{t-1} + b_h)   <-- the recurrence
            hiddenState = np.tanh(
                self.inputToHidden @ inputs[t] + self.hiddenToHidden @ hiddenState + self.hiddenBias
            )
            hiddenStates.append(hiddenState)
            # y_t = W_hy h_t + b_y                        <-- per-step output head
            logits.append(self.hiddenToOutput @ hiddenState + self.outputBias)
        return np.array(hiddenStates), np.array(logits)

rng = np.random.default_rng(1)
sequenceLength, inputSize, hiddenSize, vocabSize = 5, 4, 3, 6
inputs = rng.standard_normal((sequenceLength, inputSize))       # pretend token embeddings

rnn = SimpleRNN(inputSize, hiddenSize, vocabSize)
hiddenStates, logits = rnn.forward(inputs)

print("hidden states shape:", hiddenStates.shape, "(T, hiddenSize)")
print("logits shape       :", logits.shape, "(T, vocabSize)")
print("h_1:", np.round(hiddenStates[0], 3))
print("h_5:", np.round(hiddenStates[-1], 3), " <- summary of ALL 5 tokens")

# Parameter count does NOT depend on sequence length -- same net runs on any T
_, logitsLong = rnn.forward(rng.standard_normal((50, inputSize)))
print("same weights on T=50:", logitsLong.shape)

hidden states shape: (5, 3) (T, hiddenSize)
logits shape       : (5, 6) (T, vocabSize)
h_1: [ 0.005 -0.333 -0.648]
h_5: [-0.634 -0.702  0.441]  <- summary of ALL 5 tokens
same weights on T=50: (50, 6)


In [2]:
import numpy as np

# Vanishing / exploding gradients, empirically: how strongly does h_T still depend on h_k?
# Magnitude of  prod_{i=k+1..T} diag(1 - h_i^2) W_hh  as the distance (T - k) grows.
def gradientNormOverDistance(spectralScale: float, hiddenSize: int = 20,
                             sequenceLength: int = 40, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    recurrent = rng.standard_normal((hiddenSize, hiddenSize)) / np.sqrt(hiddenSize)
    recurrent *= spectralScale / max(abs(np.linalg.eigvals(recurrent)))   # set largest |eigenvalue|
    inputs = rng.standard_normal((sequenceLength, hiddenSize)) * 0.5

    hidden = np.zeros(hiddenSize)
    hiddens = []
    for t in range(sequenceLength):                    # forward pass, collect h_t
        hidden = np.tanh(recurrent @ hidden + inputs[t])
        hiddens.append(hidden)

    jacobian = np.eye(hiddenSize)                      # start from dh_T/dh_T = I
    norms = []
    for t in reversed(range(sequenceLength)):          # walk backward, accumulating the product
        jacobian = jacobian @ (np.diag(1 - hiddens[t]**2) @ recurrent)
        norms.append(np.linalg.norm(jacobian))
    return np.array(norms)                             # index = distance back in time

print("gradient norm  d(h_T)/d(h_k)  by distance (T - k):")
for scale, label in [(0.5, "sigma=0.5 -> VANISHING"),
                     (1.0, "sigma=1.0 -> slow decay"),
                     (2.5, "sigma=2.5 -> EXPLODING")]:
    norms = gradientNormOverDistance(scale)
    print(f"  {label:26s} 1: {norms[0]:9.2e}   10: {norms[9]:9.2e}   40: {norms[39]:9.2e}")

# Note: even sigma slightly > 1 often still decays -- the diag(1 - h^2) factors are < 1 when
# tanh saturates, so saturation damps growth. It takes a clearly large sigma to truly explode.
print("\nExponential in DISTANCE -> a plain RNN cannot learn long-range dependencies.")
print("Attention connects any two positions in ONE step (path length 1, no product at all).")

gradient norm  d(h_T)/d(h_k)  by distance (T - k):
  sigma=0.5 -> VANISHING     1:  1.74e+00   10:  3.99e-04   40:  8.60e-16
  sigma=1.0 -> slow decay    1:  3.45e+00   10:  2.15e-01   40:  1.41e-05
  sigma=2.5 -> EXPLODING     1:  5.99e+00   10:  3.52e+01   40:  2.76e+03

Exponential in DISTANCE -> a plain RNN cannot learn long-range dependencies.
Attention connects any two positions in ONE step (path length 1, no product at all).


## LSTM (Long Short-Term Memory)

### 1. Which problem are we solving?

The RNN above works, right up until the distance matters. Its gradient across `t − k` steps was

$$\frac{\partial h_t}{\partial h_k} = \prod_{i=k+1}^{t} \operatorname{diag}(1 - h_i^2)\, W_{hh}$$

a **product of `t − k` factors fixed by the weights**. Two problems follow from that single line:

- **The gradient decays exponentially** in distance, so dependencies beyond ~10-20 steps are
  effectively unlearnable. Olah's example: *"the clouds are in the ___"* is easy (the answer is
  two words back), but *"I grew up in France … I speak fluent ___"* needs information from
  dozens of steps back — and the gradient that would teach the model to use it has already died.
- **Memory is rewritten every step, unconditionally.** `h_{t-1}` goes through `W_hh` and `tanh`
  to become `h_t`. There is no way for the network to say *"leave this alone for a while"* —
  after 30 steps, anything worth keeping has been through 30 lossy transformations.

The LSTM (Hochreiter & Schmidhuber, 1997) fixes both with one idea: add a **second memory** that
is updated by **addition instead of matrix multiplication**, and let the network **learn, per step
and per dimension, what to erase, what to write, and what to read out**.

### 2. Architecture and the math behind it

An LSTM step carries **two** vectors forward, not one:

- **`C_t` — the cell state.** The long-term memory. Private: no other part of the network reads
  it directly. This is Olah's "conveyor belt" running straight through the top of the diagram.
- **`h_t` — the hidden state.** A *filtered view* of `C_t`. The short-term working output: it
  feeds the output head, and it feeds the next step's gate decisions.

Everything is controlled by three **gates**. A gate is a sigmoid layer producing a vector in
`(0, 1)`, used as an elementwise multiplier — a soft, differentiable valve. `0` = block
completely, `1` = pass through untouched.

All three gates read the same thing: the concatenation `[h_{t-1}, x_t]` (what I was thinking, plus
what I'm seeing now).

**Step 1 — forget gate: what to erase from memory.**

$$f_t = \sigma\big(W_f \cdot [h_{t-1}, x_t] + b_f\big)$$

- `σ` — sigmoid, squashing to `(0, 1)`; this is what makes `f_t` a valve rather than a value.
- `[h_{t-1}, x_t]` — concatenated previous hidden state and current input, length `d_h + d_in`.
- `W_f` ∈ ℝ^(`d_h × (d_h + d_in)`), `b_f` — the forget gate's own weights.
- `f_t` ∈ `(0,1)^{d_h}` — **one value per memory slot**. `f_t[j] ≈ 1` → keep slot `j` intact;
  `≈ 0` → wipe it. Olah's example: on seeing a new subject, forget the old subject's gender.
- **`b_f` is usually initialized positive** (e.g. `1.0`) so gates start near 1 and memory is
  retained by default at the beginning of training — a real, important practical trick.

**Step 2 — input gate + candidate: what to write.**

$$i_t = \sigma\big(W_i \cdot [h_{t-1}, x_t] + b_i\big) \qquad \tilde{C}_t = \tanh\big(W_C \cdot [h_{t-1}, x_t] + b_C\big)$$

- `i_t` ∈ `(0,1)^{d_h}` — **how much** to write into each slot (the write-enable line).
- `C̃_t` ∈ `(−1,1)^{d_h}` — **what** to write: the candidate content. `tanh`, not sigmoid,
  because content is a signed value, not a valve.
- The split matters: *how much* and *what* are decided by **separate** weights, so the network can
  compute a candidate and independently decide to ignore it.

**Step 3 — update the cell state. This is the important line.**

$$C_t = f_t \odot C_{t-1} \;+\; i_t \odot \tilde{C}_t$$

- `⊙` — **elementwise** product, so every slot is an independent little memory with its own
  erase/write decision. In one step the LSTM can forget slot 3, hold slot 7, overwrite slot 12.
- `f_t ⊙ C_{t-1}` — the retained part of the old memory.
- `i_t ⊙ C̃_t` — the newly written part.
- **What's absent is the point:** no weight matrix and no `tanh` touch `C_{t-1}`. If the network
  sets `f_t ≈ 1, i_t ≈ 0`, then `C_t = C_{t-1}` — an **exact, lossless copy**, and information
  rides the belt unchanged for hundreds of steps. The vanilla RNN cannot express this.
- **The gradient consequence:** the path from `C_t` back to `C_{t-1}` is just

$$\frac{\partial C_t}{\partial C_{t-1}} = \operatorname{diag}(f_t) \qquad\Longrightarrow\qquad \frac{\partial C_t}{\partial C_k} = \prod_{i=k+1}^{t} \operatorname{diag}(f_i)$$

  Still a product — but of **gate values the network controls**, not of a fixed `W_hh`. Hold the
  gates near 1 and the gradient arrives from 100 steps back essentially intact. This additive,
  gated highway is the *entire* reason LSTMs work. (It's the same trick as a residual connection,
  Step 5 — an unobstructed path for the gradient.)

**Step 4 — output gate: what to read out.**

$$o_t = \sigma\big(W_o \cdot [h_{t-1}, x_t] + b_o\big) \qquad h_t = o_t \odot \tanh(C_t)$$

- `tanh(C_t)` — squash the memory to `(−1,1)` before exposing it (`C` itself is unbounded, since
  it's built by accumulation).
- `o_t` — the **read-enable** line: which parts of memory are relevant *right now*.
- `h_t` is therefore only a **filtered view** of the memory. This is the second thing the cell
  state buys us: a fact can sit in `C` for 50 steps with `o_t ≈ 0` keeping it hidden, then be
  exposed at the exact moment it's needed. In a vanilla RNN, memory *is* the output — anything you
  want to remember must also be part of what you emit right now.
- The output head is then the same as the RNN's: `y_t = W_hy h_t + b_y`, softmax, cross-entropy.

**The three gates in one line each:**

| gate | formula | role | RAM analogy |
|---|---|---|---|
| `f_t` | `σ(W_f·[h_{t-1},x_t])` | what to drop from `C` | erase enable |
| `i_t`, `C̃_t` | `σ(...)`, `tanh(...)` | how much / what to store | write enable + data |
| `o_t` | `σ(W_o·[h_{t-1},x_t])` | what to expose as `h_t` | read enable |

So an LSTM is a small **differentiable memory with learned read/write/erase control**, where a
vanilla RNN is a single register that gets fully overwritten every step.

**Worked intuition.** Reading *"The **keys** that the man on the roof left ... **were** rusty"*:
one slot of `C` latches "subject is plural" at *keys* (`i≈1`), `f≈1` holds it through the whole
clause while `o≈0` keeps it out of the way, then `o` opens at *were* to get the agreement right.
A new subject appears → `f→0` for that slot → forgotten. Same story for Olah's France/French.

**The pipeline of calculations, in two scenarios.**

**(A) Learning (training).** Whole sequence known upfront (teacher forcing):

- Embed the input ids into `x_1 … x_T`; start with `h_0 = 0`, `C_0 = 0`.
- Forward loop `t = 1 … T`: from `[h_{t-1}, x_t]` compute the three gates and the candidate, then
  `C_t`, then `h_t`, then the logits `y_t`. Cache **all** of `f_t, i_t, o_t, C̃_t, C_t, h_t` —
  the backward pass needs every one (≈4× the RNN's activation memory).
- Loss: cross-entropy of `y_t` against `target_t`, averaged over all `T` positions.
- Backward loop `t = T … 1`. Each step's `C_t` collects gradient from **two** paths: through
  `h_t = o_t ⊙ tanh(C_t)` (this step's output) and straight from `C_{t+1}` scaled by `f_{t+1}` —
  the highway. The gates get gradients too, so the model *learns when to remember*.
- Accumulate into the shared weights (`W_f, W_i, W_C, W_o` and biases) — one `+=` per time step,
  same weight-sharing rule as the RNN.
- Clip the gradient norm (exploding is still possible through the gates), then one Adam step.

**(B) Actual working (generation).** Weights frozen, nothing exists yet:

- Carry **two** vectors instead of one: `h` and `C`, both starting at 0 (for captioning, seed
  them from the image features).
- Feed `<bos>`. Compute the gates from `[h, x]`, update `C` in place, then `h` from the new `C`.
- Project `h` to logits → softmax → pick a token (greedy / temperature / top-k).
- Emit it, feed it back in as the next input, repeat until `<eos>` or a length cap.
- No loss, no backward, nothing cached — only the current `(h, C)` pair.

**What differs from the RNN pipeline:** 4× the parameters and 4× the cached activations per step
(four weight matrices instead of one), two state vectors to carry instead of one, and a gradient
path that survives distance. The **sequential** structure is completely unchanged — that's the
part attention has to fix.

**Variants (from the post).**

- **Peephole connections** — let the gates also look at `C_{t-1}`, not just `[h_{t-1}, x_t]`, so
  memory content can influence the control decisions.
- **Coupled forget/input** — set `i_t = 1 − f_t`: only write to a slot when you erase it.
- **GRU** (Cho et al., 2014) — the popular simplification: merge forget+input into one **update
  gate** `z_t`, and **merge `C` and `h` into a single state**:

$$z_t = \sigma(W_z\cdot[h_{t-1},x_t]), \quad r_t = \sigma(W_r\cdot[h_{t-1},x_t])$$
$$\tilde{h}_t = \tanh\big(W\cdot[r_t \odot h_{t-1},\, x_t]\big), \qquad h_t = (1-z_t)\odot h_{t-1} + z_t \odot \tilde{h}_t$$

  - `z_t` — update gate: interpolate between keeping the old state and taking the new candidate.
  - `r_t` — reset gate: how much of the old state the candidate is even allowed to see.
  - The `(1−z)·h + z·h̃` form keeps the **additive** highway, which is what mattered. 3 weight
    matrices instead of 4, no separate `C`, usually comparable accuracy — pick either.

### 3. Applications (when to use)

Everywhere the RNN was used, but where the dependencies are actually long — which was, in
practice, everywhere. From ~2014 to ~2018 the LSTM was the default sequence model:

- **Language modeling and text generation** (Karpathy's char-RNN is an LSTM)
- **Machine translation** — seq2seq encoder/decoder LSTMs; Google Translate, 2016
- **Speech recognition**, handwriting recognition, OCR
- **Image captioning — our task's ancestor:** *Show and Tell* (2015) fed a CNN vector into an LSTM
  decoder; *Show, Attend and Tell* (2015) added attention on top of that LSTM
- **Time series**: forecasting, anomaly detection, sensor/ECG signals
- **Reinforcement learning** — an LSTM layer gives a policy memory of past observations

**Still a sound choice today** when the stream is unbounded or truly online (`O(1)` state per
token, no growing KV-cache), when latency/memory on-device is tight, or when the dataset is small
enough that a Transformer would just overfit. For anything you can batch and parallelize, use a
Transformer.

### 4. Advantages

- **Learns long-range dependencies** — the additive `C` path with `∂C_t/∂C_{t-1} = diag(f_t)`
  keeps gradients alive over hundreds of steps instead of tens.
- **Learned, per-dimension memory control** — the network decides what to keep, overwrite, and
  expose, separately for every slot and every step. No hand-designed window.
- **Separates storage from output** — a fact can be held without being emitted (`o_t ≈ 0`),
  impossible in a vanilla RNN.
- **Robust in practice** — trains reliably with modest tuning; a positive forget-gate bias is
  usually all the coaxing it needs.
- **Keeps every structural RNN advantage** — variable length, weight sharing across time, `O(1)`
  memory at generation, streaming-friendly, order built in (no positional encoding needed).

### 5. Disadvantages

- **Still strictly sequential.** `C_t` needs `C_{t-1}`. Training a length-`T` sequence is `T`
  dependent steps that a GPU cannot parallelize. **This is the reason Transformers replaced
  LSTMs** — attention computes all positions at once. Gates fixed the gradient, not the speed.
- **~4× the parameters and compute** of a vanilla RNN per step (four gate matrices), and ~4× the
  cached activations during BPTT.
- **Mitigates rather than eliminates vanishing.** `∏ f_i` still decays if the gates sit below 1 —
  a badly initialized forget gate (`f ≈ 0.5`) vanishes just as fast as a plain RNN. Exploding
  gradients also remain, so clipping is still standard.
- **The bottleneck survives.** However well `C` retains, the entire prefix is still squeezed into
  one fixed-size vector. In seq2seq this was the failure that **attention was invented to fix**.
- **Long-range in principle ≠ in practice.** Real LSTM LMs use a few hundred tokens of effective
  context, far short of a Transformer's window.
- **Hard to interpret** — no attention map. You can probe individual cell slots, but there's no
  direct readout of "what the model looked at."
- **Fiddly to implement** — four gates, two states, and a backward pass with two gradient paths
  into `C_t` is markedly more error-prone to hand-derive than attention's.

> **Where this leaves us.** The LSTM removed the *gradient* obstacle to long-range memory but kept
> the two structural ones: **sequential computation** and **one fixed-size summary vector**.
> Olah's own closing note points at the fix — let each step *"pick information to look at from
> some larger collection of information."* That is attention, and it's the next section.

### 6. Useful resources

- Olah — **Understanding LSTM Networks**
  <https://colah.github.io/posts/2015-08-Understanding-LSTMs/> — the source for this section; the
  four-step walkthrough with the conveyor-belt diagram is the clearest explanation there is.
- Karpathy — **The Unreasonable Effectiveness of RNNs**
  <http://karpathy.github.io/2015/05/21/rnn-effectiveness/> — char-LSTM in ~100 lines of NumPy,
  plus the cell-visualization section showing individual slots tracking quote-open/quote-closed.
- Hochreiter & Schmidhuber (1997) — *Long Short-Term Memory* — the original paper.
- Cho et al. (2014) — *Learning Phrase Representations…* — introduces the GRU.
- Greff et al. (2015) — *LSTM: A Search Space Odyssey* — ablates every gate; the empirical answer
  to "which parts actually matter" (the forget gate matters most).

In [ ]:
import numpy as np

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-x))

# One LSTM cell: carries TWO states forward -- cellState (C) and hiddenState (h).
class LSTMCell:
    def __init__(self, inputSize: int, hiddenSize: int, forgetBias: float = 1.0, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        self.hiddenSize: int = hiddenSize
        concatSize: int = hiddenSize + inputSize          # gates read [h_{t-1}, x_t]
        scale: float = 1 / np.sqrt(concatSize)

        self.forgetWeights: np.ndarray    = rng.standard_normal((hiddenSize, concatSize)) * scale  # W_f
        self.inputWeights: np.ndarray     = rng.standard_normal((hiddenSize, concatSize)) * scale  # W_i
        self.candidateWeights: np.ndarray = rng.standard_normal((hiddenSize, concatSize)) * scale  # W_C
        self.outputWeights: np.ndarray    = rng.standard_normal((hiddenSize, concatSize)) * scale  # W_o

        # forget bias starts POSITIVE -> f_t near 1 -> memory retained by default early in training
        self.forgetBias: np.ndarray    = np.full(hiddenSize, forgetBias)
        self.inputBias: np.ndarray     = np.zeros(hiddenSize)
        self.candidateBias: np.ndarray = np.zeros(hiddenSize)
        self.outputBias: np.ndarray    = np.zeros(hiddenSize)

    def step(self, x: np.ndarray, hiddenState: np.ndarray, cellState: np.ndarray) -> tuple:
        concatenated = np.concatenate([hiddenState, x])                              # [h_{t-1}, x_t]

        forgetGate    = sigmoid(self.forgetWeights    @ concatenated + self.forgetBias)     # f_t: what to erase
        inputGate     = sigmoid(self.inputWeights     @ concatenated + self.inputBias)      # i_t: how much to write
        candidate     = np.tanh(self.candidateWeights @ concatenated + self.candidateBias)  # C~_t: what to write
        outputGate    = sigmoid(self.outputWeights    @ concatenated + self.outputBias)     # o_t: what to expose

        newCellState   = forgetGate * cellState + inputGate * candidate   # C_t -- ADDITIVE, no matrix on C_{t-1}
        newHiddenState = outputGate * np.tanh(newCellState)               # h_t -- filtered VIEW of the memory
        return newHiddenState, newCellState, forgetGate

    def forward(self, inputs: np.ndarray) -> tuple:
        hiddenState = np.zeros(self.hiddenSize)      # h_0 = 0
        cellState   = np.zeros(self.hiddenSize)      # C_0 = 0
        hiddenStates, cellStates, forgetGates = [], [], []
        for t in range(inputs.shape[0]):
            hiddenState, cellState, forgetGate = self.step(inputs[t], hiddenState, cellState)
            hiddenStates.append(hiddenState); cellStates.append(cellState); forgetGates.append(forgetGate)
        return np.array(hiddenStates), np.array(cellStates), np.array(forgetGates)

rng = np.random.default_rng(1)
sequenceLength, inputSize, hiddenSize = 8, 4, 5
inputs = rng.standard_normal((sequenceLength, inputSize))

lstm = LSTMCell(inputSize, hiddenSize)
hiddenStates, cellStates, forgetGates = lstm.forward(inputs)

print("hidden states h:", hiddenStates.shape, " cell states C:", cellStates.shape)
print("\nmean forget gate per step (near 1 -> memory kept):", np.round(forgetGates.mean(axis=1), 3))
print("\nC grows by ACCUMULATION (unbounded), h is bounded by tanh * o:")
print("  |C| max over steps:", np.round(np.abs(cellStates).max(axis=1), 3))
print("  |h| max over steps:", np.round(np.abs(hiddenStates).max(axis=1), 3))

# The gate that HOLDS: force f=1, i=0 and the cell state is copied EXACTLY
frozen = LSTMCell(inputSize, hiddenSize, forgetBias=20.0)     # sigmoid(20) ~ 1.0
frozen.inputWeights[:] = 0; frozen.inputBias[:] = -20.0       # sigmoid(-20) ~ 0 -> write nothing
h, C = np.zeros(hiddenSize), np.array([0.7, -0.4, 0.1, 0.9, -0.2])   # a memory we want to keep
original = C.copy()
for t in range(100):
    h, C, _ = frozen.step(inputs[t % sequenceLength], h, C)
print("\nafter 100 steps with f~1, i~0 -> C unchanged:", np.allclose(C, original), np.round(C, 4))

In [ ]:
import numpy as np

# Why the cell state fixes the gradient: compare the per-step multiplier over distance.
#   vanilla RNN :  dh_t/dh_{t-1} = diag(1 - h^2) W_hh   -> fixed by the WEIGHTS
#   LSTM        :  dC_t/dC_{t-1} = diag(f_t)            -> chosen by the NETWORK
def rnnGradientNorm(distance: int, spectralScale: float, hiddenSize: int = 20, seed: int = 0) -> float:
    rng = np.random.default_rng(seed)
    recurrent = rng.standard_normal((hiddenSize, hiddenSize)) / np.sqrt(hiddenSize)
    recurrent *= spectralScale / max(abs(np.linalg.eigvals(recurrent)))
    hidden, jacobian = np.zeros(hiddenSize), np.eye(hiddenSize)
    for _ in range(distance):
        hidden = np.tanh(recurrent @ hidden + rng.standard_normal(hiddenSize) * 0.5)
        jacobian = jacobian @ (np.diag(1 - hidden**2) @ recurrent)
    return np.linalg.norm(jacobian) / np.sqrt(hiddenSize)

def lstmGradientNorm(distance: int, forgetGateValue: float) -> float:
    return forgetGateValue ** distance          # prod of diag(f) -> just f^distance per slot

print("relative gradient magnitude reaching back `distance` steps\n")
print(f"{'distance':>9} | {'RNN sigma=0.9':>14} | {'LSTM f=0.5':>11} | {'LSTM f=0.95':>12} | {'LSTM f=0.999':>13}")
print("-" * 74)
for distance in [10, 25, 50, 100, 200]:
    print(f"{distance:>9} | {rnnGradientNorm(distance, 0.9):>14.2e} | "
          f"{lstmGradientNorm(distance, 0.5):>11.2e} | {lstmGradientNorm(distance, 0.95):>12.2e} | "
          f"{lstmGradientNorm(distance, 0.999):>13.2e}")

print("\nRNN            : dead within tens of steps, and nothing can be done about it.")
print("LSTM f=0.5     : vanishes just as fast -- gates are not magic (hence the positive b_f init).")
print("LSTM f->1      : gradient still ~intact at 200 steps. The network LEARNS to hold the gate open.")